In [3]:
try:
    from databricks.sdk import WorkspaceClient
except ImportError:
    !pip install databricks-sdk
    from databricks.sdk import WorkspaceClient

try:
    import chompjs
except ImportError:
    !pip install chompjs
    import chompjs

In [4]:
# Importing Libraries
import re
import chompjs
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
from databricks.sdk import WorkspaceClient
from google.colab import userdata

# Defining functions

In [8]:
# ----- # ----- # ----- Get Jobs for CVS ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def jobs_cvs(skip=None):
    response = ''
    try:
        url = "https://jobs.cvshealth.com/us/en/search-results"
        params = {
            's': 1
        }

        if skip:
          params['from'] = skip

        headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'application/json, text/plain, */*',
        'Referer': 'https://jobs.cvshealth.com/us/en/search-results',
        'Accept-Encoding': 'gzip, deflate'
        }

        response = requests.get(url, params=params, headers=headers)
        if 'phApp' in response.text and 'eagerLoadRefineSearch' in response.text:
          return True, response.text
        else:
          return False, response.text
    except Exception as e:
        print(str(e))
        print(f"Unable to connect to CVS: {response.text}")
        return False, response.text


# ----- # ----- # ----- Fetch Job List ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def fetch_jobs():
    skip = 10
    job_df_list = []

    # Start the loop
    while True:
          if skip%100 == 0:
            print(f'Offset: {skip}, Count: {len(job_df_list)}')
          job_bool, job_text = jobs_cvs(skip)

          if job_bool:
            soup = BeautifulSoup(job_text, 'html.parser')
            script = soup.find('script', string=re.compile(r'phApp\.ddo'))

            if script:
              js_text = script.string
              start_index = js_text.find('{', js_text.find('phApp.ddo'))
              ddo_dict = chompjs.parse_js_object(js_text[start_index:])
              jobs = ddo_dict.get('eagerLoadRefineSearch', {}).get('data', {}).get('jobs', [])
              if jobs:
                job_df_list.extend(jobs)
                skip += 10
              else:
                break
            else:
              break
          else:
            break

    # Concatenate the dataset and clean
    jobs = pd.DataFrame(job_df_list)
    jobs = jobs.drop_duplicates(subset=['reqId'] ,keep='first', ignore_index=True)
    jobs.to_parquet('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/cvs_jobs.parquet', index=False)
    print(f'Total Jobs: {len(jobs)}')
    return jobs

In [9]:
# ----- # ----- # ----- Upload file to Databricks ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def upload_data(data_file):
  w = WorkspaceClient(
      host = userdata.get('DATABRICKS_HOST'),
      token = userdata.get('DATABRICKS_TOKEN')
  )

  volume_path = "/Volumes/workspace/jobs_raw/job_files/" + data_file.split('/')[-1]
  with open(data_file, "rb") as f:
      w.files.upload(volume_path, f)

  print(f"Successfully uploaded {data_file} to {volume_path}")

# New Architecture

In [23]:
job_frame = fetch_jobs()
upload_data('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/cvs_jobs.parquet')

Successfully uploaded /content/drive/MyDrive/Colab Notebooks/JobAI/jobs/cvs_jobs.parquet to /Volumes/workspace/jobs_raw/job_files/cvs_jobs.parquet
